# Figure 4 interactive viewer

Drives `statespacecheck_paper.interactive` against a pre-built cache.

## One-time cache build

Before running this notebook, build the cache from the existing decoder
intermediates:

```bash
python -m statespacecheck_paper.interactive.cache build \
    --model both --data-dir data --cache-dir data/cache
```

## Launching the viewer

Run the cell below to open the viewer in a separate native window. The
Qt event loop blocks the notebook kernel while the window is open;
close the window to return control.

## Controls

- **Center-time slider** — scrub through the session.
- **Window slider** — adjust the visible window width (log-spaced,
  0.1 s to 60 s).
- **Likelihood α** — opacity of the likelihood fill in the slice panel.
- **Play / pause** (`Space`) — auto-scroll at 1 × realtime.
- **Model dropdown** (`M`) — toggle between Continuous and ContFrag.
- **Click on raster spike or metric dot** — recenter the window on
  that spike's time and pin a marker across all panels (slice panel
  shows the cell's place field plus an annotation with the per-spike
  metrics).
- **Keyboard**: `←`/`→` step one bin; `Shift+←`/`Shift+→` step one
  window; `[`/`]` shrink/grow window; `R` reset to a 20 s context
  window.

In [ ]:
from pathlib import Path

from statespacecheck_paper.interactive.viewer import launch

REPO_ROOT = Path().resolve().parent
CACHE_DIR = REPO_ROOT / "data" / "cache"
assert CACHE_DIR.is_dir(), f"Cache missing: {CACHE_DIR}"

# Pick the model to start with; switch live via the dropdown.
launch(CACHE_DIR, model="continuous")

## Programmatic access without launching the viewer

If you want to script the cache (e.g. extract one window of
posterior + likelihood + events for offline plotting), use
`Figure4DataSource` directly.

In [ ]:
from statespacecheck_paper.interactive.data_source import Figure4DataSource

with Figure4DataSource(CACHE_DIR, model="continuous") as ds:
    sl = ds.window_indices(t_center=ds.time[100_000], t_width=2.0)
    posterior = ds.load_posterior(sl)
    likelihood = ds.load_likelihood(sl)
    events = ds.events_in_window(sl)
    print("posterior:", posterior.shape, posterior.dtype)
    print("likelihood:", likelihood.shape, likelihood.dtype)
    print("events:", events.shape)